In [ ]:
# ============================================================
# PROJECT 7: ATTENTION-BASED ENGLISH–TAMIL TRANSLATOR
# PyTorch + English-Tamil Parallel Dataset
# ============================================================

# If libraries are missing, run this once:
# !pip install pandas numpy torch matplotlib scikit-learn


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import random
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from collections import Counter


# ============================================================
# 2. RANDOM SEED
# ============================================================

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


# ============================================================
# 3. SELECT DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ============================================================
# 4. LOAD DATASET
# ============================================================

DATASET = "final_translation.csv"

df = pd.read_csv(
    DATASET,
    encoding="utf-8-sig"
)

print("\nDataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())


# ============================================================
# 5. FIND ENGLISH AND TAMIL COLUMNS
# ============================================================

english_column = None
tamil_column = None

for col in df.columns:

    col_lower = str(col).lower()

    if (
        "english" in col_lower
        or col_lower == "en"
    ):
        english_column = col

    if (
        "tamil" in col_lower
        or col_lower == "ta"
    ):
        tamil_column = col


# If columns cannot be detected,
# use the first two columns.

if english_column is None or tamil_column is None:

    print(
        "\nEnglish/Tamil column names were not automatically detected."
    )

    print(
        "Using the first two columns as English and Tamil."
    )

    english_column = df.columns[0]
    tamil_column = df.columns[1]


print("\nEnglish column:", english_column)
print("Tamil column  :", tamil_column)


# ============================================================
# 6. KEEP REQUIRED COLUMNS
# ============================================================

df = df[
    [english_column, tamil_column]
].copy()

df.columns = [
    "english",
    "tamil"
]


# ============================================================
# 7. REMOVE MISSING VALUES
# ============================================================

df = df.dropna()

df["english"] = df["english"].astype(str)
df["tamil"] = df["tamil"].astype(str)

df = df[
    (df["english"].str.strip() != "")
    &
    (df["tamil"].str.strip() != "")
]

df = df.drop_duplicates()

print("\nDataset after cleaning:")
print("Number of sentence pairs:", len(df))


# ============================================================
# 8. OPTIONAL: LIMIT DATASET FOR FAST LAPTOP TRAINING
# ============================================================

# Set this to None if you want to use the complete dataset.
# 1000 pairs is enough for a classroom demonstration
# and trains much faster.

MAX_PAIRS = 1000


# Keep the required example "How are you?" if available.

demo_rows = df[
    df["english"]
    .str.strip()
    .str.lower()
    == "how are you?"
]

other_rows = df[
    ~(
        df["english"]
        .str.strip()
        .str.lower()
        == "how are you?"
    )
]


if MAX_PAIRS is not None and len(df) > MAX_PAIRS:

    required_count = min(
        len(demo_rows),
        1
    )

    remaining_count = MAX_PAIRS - required_count

    if remaining_count > 0:

        other_rows = other_rows.sample(
            n=min(
                remaining_count,
                len(other_rows)
            ),
            random_state=42
        )

    else:

        other_rows = other_rows.iloc[0:0]

    df = pd.concat(
        [
            demo_rows.head(1),
            other_rows
        ],
        ignore_index=True
    )


print(
    "\nPairs used for training/testing:",
    len(df)
)


# ============================================================
# 9. CLEAN TEXT
# ============================================================

def clean_english(text):

    text = str(text).lower()

    # Separate punctuation
    text = re.sub(
        r"([?.!,])",
        r" \1 ",
        text
    )

    # Remove unwanted characters
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def clean_tamil(text):

    text = str(text)

    # Separate punctuation
    text = re.sub(
        r"([?.!,])",
        r" \1 ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df["english_clean"] = df[
    "english"
].apply(clean_english)

df["tamil_clean"] = df[
    "tamil"
].apply(clean_tamil)


# ============================================================
# 10. TOKENIZATION
# ============================================================

df["english_tokens"] = df[
    "english_clean"
].apply(
    lambda x: x.split()
)

df["tamil_tokens"] = df[
    "tamil_clean"
].apply(
    lambda x: x.split()
)


print("\nExample tokenization:")

print(
    "English:",
    df.iloc[0]["english_tokens"]
)

print(
    "Tamil:",
    df.iloc[0]["tamil_tokens"]
)


# ============================================================
# 11. TRAIN / TEST SPLIT
# ============================================================

# Keep "How are you?" in training so the required
# demonstration can be learned.

demo_mask = (
    df["english_clean"]
    == "how are you ?"
)

demo_df = df[
    demo_mask
].copy()

other_df = df[
    ~demo_mask
].copy()


# Shuffle other sentences

other_df = other_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# 80% training and 20% testing

test_size = max(
    1,
    int(len(other_df) * 0.20)
)


test_df = other_df.iloc[
    :test_size
].copy()


train_df = other_df.iloc[
    test_size:
].copy()


# Add demo sentence to training

if len(demo_df) > 0:

    train_df = pd.concat(
        [
            train_df,
            demo_df
        ],
        ignore_index=True
    )


# Shuffle training data

train_df = train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


print("\nDataset split:")
print(
    "Training sentences:",
    len(train_df)
)

print(
    "Testing sentences :",
    len(test_df)
)


# ============================================================
# 12. SPECIAL TOKENS
# ============================================================

PAD = "<PAD>"
START = "<START>"
END = "<END>"
UNK = "<UNK>"

SPECIAL_TOKENS = [
    PAD,
    START,
    END,
    UNK
]


# ============================================================
# 13. BUILD VOCABULARY
# ============================================================

def build_vocab(token_lists):

    counter = Counter()

    for tokens in token_lists:
        counter.update(tokens)

    vocab = {}

    for token in SPECIAL_TOKENS:
        vocab[token] = len(vocab)

    for word, count in counter.items():

        if word not in vocab:

            vocab[word] = len(vocab)

    return vocab


english_vocab = build_vocab(
    train_df["english_tokens"]
)

tamil_vocab = build_vocab(
    train_df["tamil_tokens"]
)


english_id_to_word = {
    value: key
    for key, value in english_vocab.items()
}


tamil_id_to_word = {
    value: key
    for key, value in tamil_vocab.items()
}


print("\nVocabulary sizes:")

print(
    "English vocabulary:",
    len(english_vocab)
)

print(
    "Tamil vocabulary:",
    len(tamil_vocab)
)


# ============================================================
# 14. CONVERT WORDS TO NUMBERS
# ============================================================

def tokens_to_ids(
    tokens,
    vocab,
    add_start_end=False
):

    ids = []

    if add_start_end:

        ids.append(
            vocab[START]
        )

    for token in tokens:

        ids.append(
            vocab.get(
                token,
                vocab[UNK]
            )
        )

    if add_start_end:

        ids.append(
            vocab[END]
        )

    return ids


# ============================================================
# 15. CREATE NUMERICAL SEQUENCES
# ============================================================

train_source_sequences = []

train_target_sequences = []

test_source_sequences = []

test_target_sequences = []


for _, row in train_df.iterrows():

    source_ids = tokens_to_ids(
        row["english_tokens"],
        english_vocab,
        add_start_end=False
    )

    target_ids = tokens_to_ids(
        row["tamil_tokens"],
        tamil_vocab,
        add_start_end=True
    )

    train_source_sequences.append(
        source_ids
    )

    train_target_sequences.append(
        target_ids
    )


for _, row in test_df.iterrows():

    source_ids = tokens_to_ids(
        row["english_tokens"],
        english_vocab,
        add_start_end=False
    )

    target_ids = tokens_to_ids(
        row["tamil_tokens"],
        tamil_vocab,
        add_start_end=True
    )

    test_source_sequences.append(
        source_ids
    )

    test_target_sequences.append(
        target_ids
    )


# ============================================================
# 16. PAD SEQUENCES
# ============================================================

def pad_sequences(
    sequences,
    pad_id
):

    max_length = max(
        len(seq)
        for seq in sequences
    )

    padded = []

    for seq in sequences:

        padded_seq = (
            seq
            +
            [pad_id] *
            (max_length - len(seq))
        )

        padded.append(
            padded_seq
        )

    return torch.tensor(
        padded,
        dtype=torch.long
    )


train_source = pad_sequences(
    train_source_sequences,
    english_vocab[PAD]
)

train_target = pad_sequences(
    train_target_sequences,
    tamil_vocab[PAD]
)


test_source = pad_sequences(
    test_source_sequences,
    english_vocab[PAD]
)

test_target = pad_sequences(
    test_target_sequences,
    tamil_vocab[PAD]
)


print("\nTensor shapes:")

print(
    "Training English:",
    train_source.shape
)

print(
    "Training Tamil  :",
    train_target.shape
)

print(
    "Testing English :",
    test_source.shape
)

print(
    "Testing Tamil   :",
    test_target.shape
)


# Move data to device

train_source = train_source.to(device)
train_target = train_target.to(device)

test_source = test_source.to(device)
test_target = test_target.to(device)


# ============================================================
# 17. ENCODER
# ============================================================

class Encoder(nn.Module):

    def __init__(
        self,
        input_size,
        embedding_size,
        hidden_size
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            input_size,
            embedding_size,
            padding_idx=english_vocab[PAD]
        )

        self.lstm = nn.LSTM(
            embedding_size,
            hidden_size,
            batch_first=True
        )


    def forward(self, source):

        embedded = self.embedding(
            source
        )

        outputs, (hidden, cell) = self.lstm(
            embedded
        )

        return (
            outputs,
            hidden,
            cell
        )


# ============================================================
# 18. ATTENTION MECHANISM
# ============================================================

class Attention(nn.Module):

    def __init__(
        self,
        hidden_size
    ):

        super().__init__()

        self.attention = nn.Linear(
            hidden_size * 2,
            hidden_size
        )

        self.score = nn.Linear(
            hidden_size,
            1,
            bias=False
        )


    def forward(
        self,
        decoder_hidden,
        encoder_outputs,
        mask
    ):

        source_length = (
            encoder_outputs.size(1)
        )

        # [batch, hidden]
        decoder_hidden = decoder_hidden.unsqueeze(1)

        # [batch, source_length, hidden]
        decoder_hidden = decoder_hidden.repeat(
            1,
            source_length,
            1
        )

        combined = torch.cat(
            [
                decoder_hidden,
                encoder_outputs
            ],
            dim=2
        )

        energy = torch.tanh(
            self.attention(
                combined
            )
        )

        scores = self.score(
            energy
        ).squeeze(2)

        # Ignore PAD words

        scores = scores.masked_fill(
            mask == 0,
            -1e10
        )

        attention_weights = torch.softmax(
            scores,
            dim=1
        )

        context = torch.bmm(
            attention_weights.unsqueeze(1),
            encoder_outputs
        )

        context = context.squeeze(1)

        return (
            context,
            attention_weights
        )


# ============================================================
# 19. DECODER
# ============================================================

class Decoder(nn.Module):

    def __init__(
        self,
        output_size,
        embedding_size,
        hidden_size
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            output_size,
            embedding_size,
            padding_idx=tamil_vocab[PAD]
        )

        self.attention = Attention(
            hidden_size
        )

        self.lstm = nn.LSTM(
            embedding_size + hidden_size,
            hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size * 2,
            output_size
        )


    def forward(
        self,
        input_token,
        hidden,
        cell,
        encoder_outputs,
        mask
    ):

        # Convert token to embedding

        embedded = self.embedding(
            input_token
        ).unsqueeze(1)

        # Current decoder hidden state

        decoder_hidden = hidden[-1]

        # Attention

        context, attention_weights = self.attention(
            decoder_hidden,
            encoder_outputs,
            mask
        )

        context = context.unsqueeze(1)

        # Combine embedding + context

        lstm_input = torch.cat(
            [
                embedded,
                context
            ],
            dim=2
        )

        # Decoder LSTM

        output, (hidden, cell) = self.lstm(
            lstm_input,
            (hidden, cell)
        )

        output = output.squeeze(1)

        context = context.squeeze(1)

        # Prediction

        prediction_input = torch.cat(
            [
                output,
                context
            ],
            dim=1
        )

        prediction = self.fc(
            prediction_input
        )

        return (
            prediction,
            hidden,
            cell,
            attention_weights
        )


# ============================================================
# 20. SEQ2SEQ MODEL
# ============================================================

class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder,
        device
    ):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device


    def forward(
        self,
        source,
        target,
        teacher_forcing_ratio=0.5
    ):

        batch_size = source.size(0)

        target_length = target.size(1)

        output_size = len(
            tamil_vocab
        )

        outputs = torch.zeros(
            batch_size,
            target_length,
            output_size
        ).to(self.device)

        # Encoder

        encoder_outputs, hidden, cell = self.encoder(
            source
        )

        # Mask PAD words

        mask = (
            source
            !=
            english_vocab[PAD]
        )

        # First decoder input is START

        input_token = target[:, 0]

        # Decoder

        for t in range(
            1,
            target_length
        ):

            prediction, hidden, cell, _ = self.decoder(
                input_token,
                hidden,
                cell,
                encoder_outputs,
                mask
            )

            outputs[:, t] = prediction

            best_prediction = prediction.argmax(
                1
            )

            teacher_force = (
                random.random()
                <
                teacher_forcing_ratio
            )

            if teacher_force:

                input_token = target[:, t]

            else:

                input_token = best_prediction

        return outputs


# ============================================================
# 21. CREATE MODEL
# ============================================================

INPUT_SIZE = len(
    english_vocab
)

OUTPUT_SIZE = len(
    tamil_vocab
)

EMBEDDING_SIZE = 64

HIDDEN_SIZE = 128


encoder = Encoder(
    INPUT_SIZE,
    EMBEDDING_SIZE,
    HIDDEN_SIZE
)

decoder = Decoder(
    OUTPUT_SIZE,
    EMBEDDING_SIZE,
    HIDDEN_SIZE
)


model = Seq2Seq(
    encoder,
    decoder,
    device
).to(device)


print("\nModel created successfully!")


# ============================================================
# 22. LOSS FUNCTION AND OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(
    ignore_index=tamil_vocab[PAD]
)


# ============================================================
# 23. TRAINING FUNCTION
# ============================================================

def train_model():

    model.train()

    optimizer.zero_grad()

    output = model(
        train_source,
        train_target,
        teacher_forcing_ratio=0.5
    )

    output_size = output.size(-1)

    output = output[:, 1:].reshape(
        -1,
        output_size
    )

    target = train_target[:, 1:].reshape(
        -1
    )

    loss = criterion(
        output,
        target
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    optimizer.step()

    return loss.item()


# ============================================================
# 24. VALIDATION / TEST LOSS
# ============================================================

def evaluate_model():

    model.eval()

    with torch.no_grad():

        output = model(
            test_source,
            test_target,
            teacher_forcing_ratio=0
        )

        output_size = output.size(-1)

        output = output[:, 1:].reshape(
            -1,
            output_size
        )

        target = test_target[:, 1:].reshape(
            -1
        )

        loss = criterion(
            output,
            target
        )

    return loss.item()


# ============================================================
# 25. TRAIN THE MODEL
# ============================================================

EPOCHS = 100

train_losses = []

validation_losses = []


print("\nStarting training...\n")


for epoch in range(
    EPOCHS
):

    train_loss = train_model()

    validation_loss = evaluate_model()

    train_losses.append(
        train_loss
    )

    validation_losses.append(
        validation_loss
    )

    if (
        (epoch + 1) % 10 == 0
        or epoch == 0
    ):

        print(
            f"Epoch {epoch + 1:3d}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Validation Loss: {validation_loss:.4f}"
        )


print("\nTraining completed!")


# ============================================================
# 26. PLOT TRAINING AND VALIDATION LOSS
# ============================================================

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    train_losses,
    label="Training Loss"
)

plt.plot(
    validation_losses,
    label="Validation Loss"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "Training and Validation Loss"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ============================================================
# 27. TRANSLATION FUNCTION
# ============================================================

def translate_sentence(
    sentence,
    max_length=20
):

    model.eval()

    # Clean input

    cleaned = clean_english(
        sentence
    )

    # Tokenize

    tokens = cleaned.split()

    # Convert to IDs

    token_ids = []

    for token in tokens:

        token_ids.append(
            english_vocab.get(
                token,
                english_vocab[UNK]
            )
        )

    # Create source tensor

    source = torch.tensor(
        [token_ids],
        dtype=torch.long
    ).to(device)

    # Encoder

    with torch.no_grad():

        encoder_outputs, hidden, cell = model.encoder(
            source
        )

    # Source mask

    mask = (
        source
        !=
        english_vocab[PAD]
    )

    # Start decoder

    input_token = torch.tensor(
        [tamil_vocab[START]],
        dtype=torch.long
    ).to(device)

    translated_words = []

    attention_values = []


    # Generate Tamil words

    for _ in range(
        max_length
    ):

        with torch.no_grad():

            prediction, hidden, cell, attention_weights = model.decoder(
                input_token,
                hidden,
                cell,
                encoder_outputs,
                mask
            )

        # Get most likely word

        best_token = prediction.argmax(
            1
        ).item()

        # Store attention

        attention_values.append(
            attention_weights[
                0
            ].cpu().numpy()
        )

        # Stop at END

        if best_token == tamil_vocab[END]:

            break

        # Stop at PAD

        if best_token == tamil_vocab[PAD]:

            break

        word = tamil_id_to_word.get(
            best_token,
            UNK
        )

        translated_words.append(
            word
        )

        # Next decoder input

        input_token = torch.tensor(
            [best_token],
            dtype=torch.long
        ).to(device)


    attention_matrix = np.array(
        attention_values
    )

    translation = " ".join(
        translated_words
    )

    return (
        translation,
        attention_matrix,
        tokens
    )


# ============================================================
# 28. REQUIRED DEMONSTRATION
# ============================================================

input_sentence = "How are you?"


translation, attention, source_words = translate_sentence(
    input_sentence
)


print("\n")
print("=" * 60)

print(
    "English Input :",
    input_sentence
)

print(
    "Tamil Output  :",
    translation
)

print("=" * 60)


# ============================================================
# 29. ATTENTION VISUALIZATION
# ============================================================

def show_attention(
    source_words,
    attention_matrix,
    translated_sentence
):

    target_words = (
        translated_sentence
        .split()
    )

    if (
        len(target_words) == 0
        or attention_matrix.size == 0
    ):

        print(
            "Attention visualization is not available."
        )

        return


    rows = min(
        len(target_words),
        attention_matrix.shape[0]
    )

    cols = min(
        len(source_words),
        attention_matrix.shape[1]
    )


    attention_matrix = attention_matrix[
        :rows,
        :cols
    ]


    target_words = target_words[
        :rows
    ]

    source_words = source_words[
        :cols
    ]


    plt.figure(
        figsize=(10, 6)
    )

    plt.imshow(
        attention_matrix,
        aspect="auto"
    )

    plt.xticks(
        range(len(source_words)),
        source_words,
        rotation=45
    )

    plt.yticks(
        range(len(target_words)),
        target_words
    )

    plt.xlabel(
        "English Source Words"
    )

    plt.ylabel(
        "Generated Tamil Words"
    )

    plt.title(
        "Attention Visualization"
    )

    plt.colorbar(
        label="Attention Weight"
    )

    plt.tight_layout()

    plt.show()


show_attention(
    source_words,
    attention,
    translation
)


# ============================================================
# 30. TEST ON UNSEEN SENTENCES
# ============================================================

print("\n")
print("=" * 60)
print("TRANSLATION RESULTS ON TEST SENTENCES")
print("=" * 60)


# Use actual unseen sentences from the test dataset

number_of_tests = min(
    5,
    len(test_df)
)


for i in range(
    number_of_tests
):

    english_sentence = (
        test_df.iloc[i]["english"]
    )

    actual_tamil = (
        test_df.iloc[i]["tamil"]
    )

    predicted_tamil, _, _ = translate_sentence(
        english_sentence
    )

    print("\nEnglish :", english_sentence)

    print(
        "Actual Tamil:",
        actual_tamil
    )

    print(
        "Predicted Tamil:",
        predicted_tamil
    )


# ============================================================
# 31. USER INPUT TRANSLATOR
# ============================================================

print("\n")
print("=" * 60)
print("ENGLISH → TAMIL TRANSLATOR")
print("=" * 60)

user_sentence = input(
    "\nEnter an English sentence: "
)


result, user_attention, user_source_words = translate_sentence(
    user_sentence
)


print(
    "\nTamil Translation:"
)

print(
    result
)


# Show attention for user input

show_attention(
    user_source_words,
    user_attention,
    result
)


# ============================================================
# END OF PROJECT
# ============================================================

Using device: cpu

Dataset loaded successfully!
Dataset shape: (11104, 2)

Column names:
['English', 'Tamil']

First 5 rows:
                                            English  \
0                              Let's try something.   
1                            I have to go to sleep.   
2  Today is June 18th and it is Muiriel's birthday!   
3                                Muiriel is 20 now.   
4                        The password is "Muiriel".   

                                          Tamil  
0                     ஏதாவது முயற்சி செய்யலாம்.  
1                    நான் தூங்க செல்ல வேண்டும்.  
2  இன்று ஜூன் 18 மற்றும் முரியலின் பிறந்த நாள்!  
3                  முரியலுக்கு இப்போது 20 வயது.  
4                         கடவுச்சொல் "முரியல்".  

English column: English
Tamil column  : Tamil

Dataset after cleaning:
Number of sentence pairs: 11104

Pairs used for training/testing: 1000

Example tokenization:
English: ['progress', 'in', 'science', 'was', 'often', 'barred', 'by', 'conven